# Chapter 6.10: Build Your First Network in Keras

Back to the course site: https://neel429.github.io/ml-for-robotics-/

This notebook repeats the two-moons experiment from the browser chapter using TensorFlow/Keras. Start with the intentionally bad baseline, then fix one knob at a time.

## Cell 1 - Generate data

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X = StandardScaler().fit_transform(X)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}  Val: {X_val.shape}")

## Cell 2 - Build the baseline, intentionally bad

In [ ]:
import tensorflow as tf
from tensorflow import keras

LEARNING_RATE = 5.0
EPOCHS = 5
HIDDEN_UNITS = 2
NUM_LAYERS = 1

model = keras.Sequential()
model.add(keras.layers.Input(shape=(2,)))
for _ in range(NUM_LAYERS):
    model.add(keras.layers.Dense(HIDDEN_UNITS, activation="relu"))
model.add(keras.layers.Dense(1, activation="sigmoid"))

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## Cell 3 - Train and plot

Run this with the broken baseline, then change the configuration in Cell 2 and rerun Cells 2 and 3.

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=32,
    verbose=1,
)

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["loss"], label="Train loss")
ax1.plot(history.history["val_loss"], label="Val loss", linestyle="--")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(history.history["accuracy"], label="Train acc")
ax2.plot(history.history["val_accuracy"], label="Val acc", linestyle="--")
ax2.set_title("Accuracy")
ax2.set_ylim(0, 1)
ax2.legend()

plt.suptitle(f"LR={LEARNING_RATE} | Epochs={EPOCHS} | {NUM_LAYERS}x{HIDDEN_UNITS} hidden units")
plt.show()

print(f"Final val accuracy: {history.history['val_accuracy'][-1]*100:.1f}%")

## Cell 4 - Fix it progressively

Rerun Cells 2 and 3 after each change.

1. Change `LEARNING_RATE = 5.0` to `0.1`.
2. Change `EPOCHS = 5` to `100`.
3. Change `HIDDEN_UNITS = 2` to `16` and `NUM_LAYERS = 1` to `2`.
4. Try over-engineering with `HIDDEN_UNITS = 256` and `NUM_LAYERS = 3`.

## Cell 5 - Add dropout and early stopping

In [ ]:
LEARNING_RATE = 0.001
EPOCHS = 200
HIDDEN_UNITS = 64
NUM_LAYERS = 2
DROPOUT_RATE = 0.3

model2 = keras.Sequential()
model2.add(keras.layers.Input(shape=(2,)))
for _ in range(NUM_LAYERS):
    model2.add(keras.layers.Dense(HIDDEN_UNITS, activation="relu"))
    model2.add(keras.layers.Dropout(DROPOUT_RATE))
model2.add(keras.layers.Dense(1, activation="sigmoid"))

model2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
)

history2 = model2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)

stopped_at = len(history2.history["loss"])
print(f"Stopped at epoch {stopped_at} / {EPOCHS}")
print(f"Final val accuracy: {history2.history['val_accuracy'][-1]*100:.1f}%")